In [1]:
import pandas as pd 
import sqlalchemy as sa
from datetime import datetime, timedelta, timezone

In [2]:
# Proses get connection to database
def connect_to_database(**params):
    username = params.get('username')
    password = params.get('password')
    host = params.get('host')
    port = params.get('port')
    database = params.get('database')
    
    engine = sa.create_engine(
        f"postgresql+psycopg://{username}:{password}@{host}:{port}/{database}"
    )
    connection = engine.connect()

    return(engine, connection)

# Proses ekstraksi data di database
def extract_data_from_database(query, conn):
    data = pd.read_sql(query, conn)
    return(data)

# Proses menambahkan metadata
def add_metadata(data):
    # Get date now
    gmt_plus_7 = timezone(timedelta(hours=14))
    current_time = datetime.now(gmt_plus_7)

    # Tambahkan kolom created date dan isikan dengan tanggal dan waktu hari ini
    data['created_date'] = current_time
    data['created_by'] = 'SYSTEM'
    return(data)

# Proses load data / menyimpan data ke database
def load_data(data, table_name, connection, insert_method):
    data.to_sql(
        table_name,
        connection,
        if_exists = insert_method,
        index = False
    )

In [8]:
query = """
    SELECT DISTINCT
        employee_id,
        employee_name,
        employee_address,
        employee_phone_number
    FROM mst_employee
"""

# Koneksi ke db_staging (bronze layer)
engine_staging, con_staging = connect_to_database(
    username = 'postgres',
    password = '',
    host = 'localhost',
    port = '5432',
    database = 'db_pizza_staging'
)

src_inspector = sa.inspect(engine_staging)
src_table_name = 'mst_employee'

src_datatype = pd.DataFrame(src_inspector.get_columns(src_table_name))
src_datatype.columns = src_datatype.columns + '_src'
display(src_datatype)

,name_src,type_src,nullable_src,default_src,autoincrement_src,comment_src
0,employee_id,VARCHAR(25),True,None,False,None
1,employee_name,VARCHAR(100),False,None,False,None
2,employee_address,TEXT,True,None,False,None
3,employee_phone_number,VARCHAR(20),True,None,False,None


In [4]:
# Proses ekstrak data
data_extracted = extract_data_from_database(query, con_staging)
data_extracted = add_metadata(data_extracted)

In [9]:
# Koneksi ke db_mart (golden layer)
engine_mart, con_mart = connect_to_database(
    username = 'postgres',
    password = '',
    host = 'localhost',
    port = '5432',
    database = 'dm_pizza_transaction'
)

dest_inspector = sa.inspect(engine_mart)
dest_table_name = 'd_employee'

dest_datatype = pd.DataFrame(dest_inspector.get_columns(dest_table_name))
dest_datatype.columns = dest_datatype.columns + '_dest'
display(dest_datatype)

,name_dest,type_dest,nullable_dest,default_dest,autoincrement_dest,comment_dest
0,employee_id,INTEGER,True,None,False,None
1,employee_name,VARCHAR(100),False,None,False,None
2,employee_address,TEXT,True,None,False,None
3,employee_phone_number,VARCHAR(20),True,None,False,None
4,created_date,TIMESTAMP,True,None,False,None
5,created_by,VARCHAR(50),True,None,False,None


In [6]:
load_data(
    data = data_extracted, 
    table_name = 'd_employee', 
    connection = con_mart,
    insert_method = 'append'
)

ProgrammingError: (psycopg.errors.DatatypeMismatch) column "employee_id" is of type integer but expression is of type character varying
LINE 1: ...e_phone_number, created_date, created_by) VALUES ($1::VARCHA...
                                                             ^
HINT:  You will need to rewrite or cast the expression.
[SQL: INSERT INTO d_employee (employee_id, employee_name, employee_address, employee_phone_number, created_date, created_by) VALUES (%(employee_id)s::VARCHAR, %(employee_name)s::VARCHAR, %(employee_address)s::VARCHAR, %(employee_phone_number)s::VARCHAR, %(created_date)s::TIMESTAMP WITH TIME ZONE, %(created_by)s::VARCHAR)]
[parameters: [{'employee_id': '208', 'employee_name': 'Rian Pratama', 'employee_address': 'Jl. Gajah Mada No. 101, Yogyakarta', 'employee_phone_number': '082134567890', 'created_date': datetime.datetime(2026, 9, 23, 3, 30, 0, 448194, tzinfo=datetime.timezone(datetime.timedelta(seconds=50400))), 'created_by': 'SYSTEM'}, {'employee_id': '295', 'e